In [6]:
# 必要なモジュールをインポート
import os
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver


# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]


# ===== グラフの構築 =====
def build_graph(model_name: str):
    # 検索ツールの定義
    tool = TavilySearchResults(max_results=2)
    tools = [tool]

    graph_builder = StateGraph(State)

    llm = ChatOpenAI(model_name=model_name)
    llm_with_tools = llm.bind_tools(tools)

    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))

    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    graph_builder.set_entry_point("chatbot")

    memory = MemorySaver()
    return graph_builder.compile(checkpointer=memory)


# ===== グラフ実行関数 =====
def stream_graph_updates(graph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values",
    )
    for event in events:
        last_message = event["messages"][-1]
        if getattr(last_message, "type", None) == "tool":
            continue
        print(last_message.content, flush=True)


# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
# ソースコードを記述
# チャットボットのループ
while True:
    user_input = input("質問: ")
    if user_input.strip() == "":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんにちは
こんにちは！今日はどんなことをお手伝いできますか？
1足す2は？
1足す2は3です。何か他にお手伝いできることがありますか？
台湾観光について検索結果を教えて

台湾観光に関する情報をいくつかご紹介します。

1. **台湾観光の達人が教える旅行術**
   - この書籍では、台湾の観光スポットやグルメ、ショッピングについての情報が詳しく紹介されています。特に、故宮博物院や夜市、温泉など、台湾の魅力的なスポットを効率よく楽しむためのテクニックが掲載されています。詳細は[こちら](http://www.horei.com/book_4-89346-746-8.html)で確認できます。

2. **台北観光のおすすめスポット**
   - 台湾は日本から直行便で約4時間で行ける人気の観光地です。台北には、国立故宮博物院や中正紀念堂、士林観光夜市など多くの観光名所があります。また、九份のような風光明媚な町も近くにあり、日帰り旅行にも最適です。詳細な情報は[こちら](https://www.knt.co.jp/travelguide/kaigai/027/)で確認できます。

これらの情報を参考に、台湾観光を楽しんでください！他に知りたいことがあれば教えてください。
ありがとうございました!
